# Tome C1, Chapter 2: Orders (Python companion)

Exercises first, solutions at the end. Posets become directed acyclic graphs: we sort them topologically, count their linear extensions, compute their width with a matching algorithm (Dilworth), and find long monotone subsequences with patience sorting, which leads to the surprising $2\sqrt n$ law for random permutations. We evaluate recursive queries as least fixed points (Knaster–Tarski and Kleene), watch terminating processes that take astronomically long (Ackermann, rewriting, the Hydra), and explore the Boolean lattice: symmetric chains, Dedekind numbers and intersecting families.

*Before each solution, predict what the output will look like.*

In [ ]:
import itertools, random, time, bisect, math
from collections import defaultdict, deque
from functools import lru_cache
import numpy as np
import matplotlib.pyplot as plt
random.seed(2)

## Exercises

**P2.1 (★) Topological sorting and linear extensions.** Generate a random poset on $n$ elements (a random DAG on $0,\dots,n-1$ with arcs $i\to j$ only for $i<j$, then its transitive closure). Implement Kahn's algorithm (repeatedly output an element with no remaining predecessor) and check that the result is a linear extension (Proposition 2.2). Count linear extensions exactly by dynamic programming over the subsets already placed, and check Exercise 2.7: the poset $\{1,2\}\times\{1,\dots,n\}$ has Catalan-many linear extensions.

**P2.2 (★★) Width by matching.** For a poset $P$, build the bipartite graph with two copies $x^-,x^+$ of each element and an edge $x^-y^+$ when $x<y$. Show by experiment that the minimum number of chains covering $P$ is $|P|$ minus the size of a maximum matching (a matching edge $x^-y^+$ means "$y$ follows $x$ in its chain"; Chapter 6 proves it). Implement augmenting-path matching, return the chains, find an antichain of the same size by brute force on small posets, and confirm Dilworth's theorem on $200$ random posets.

**P2.3 (★★) Patience sorting and the $2\sqrt n$ law.** Deal the numbers one at a time onto piles: put each on the leftmost pile whose top is larger, or start a new pile on the right. Show that the number of piles is the length of the longest increasing subsequence, implement it in $O(n\log n)$ with a binary search, recover an actual subsequence, and check the Erdős–Szekeres theorem on random sequences of length $n^2+1$. Then plot $\mathrm{LIS}/\sqrt n$ for random permutations of size up to $10^6$. What constant appears?

**P2.4 (★) Recursive queries as least fixed points.** Given a random "parent" relation on $2000$ people, compute the "ancestor" relation as the least fixed point of $X\mapsto \mathrm{parent}\cup(\mathrm{parent}\circ X)$ by Kleene iteration (Proposition 2.6), first naively, then *semi-naively* (at each round, only join the facts that are new). Compare the number of rounds and the work done.

**P2.5 (★★) Termination takes time.** (a) Compute Ackermann's function with an explicit stack (the recursion is too deep for Python) and count the calls for $A(2,n)$ and $A(3,n)$, $n\le6$. (b) Simulate the rewriting $ba\to aab$ of Exercise 2.4 from $b^ka$ with random choices and check that it always takes $2^k-1$ steps. (c) Simulate the Hydra of Exercise 2.5 with Hercules always cutting the rightmost head, on small hydras, and count the steps.

**P2.6 (★★) The Boolean lattice.** Build the symmetric chain decomposition of Theorem 2.17 for $n\le12$ and check that it is a partition into $\binom n{\lfloor n/2\rfloor}$ symmetric saturated chains. Then compute the Dedekind numbers $D(n)$ (Exercise 2.8) for $n\le6$ using the recursion "a monotone function on $B_n$ is a pair $f_0\le f_1$ of monotone functions on $B_{n-1}$", with functions stored as bit masks.

**P2.7 (★★) LYM and Erdős–Ko–Rado by computer.** (a) Build random maximal antichains in $B_8$ greedily and check the LYM inequality (Theorem 2.16). Which antichains make the sum close to $1$? (b) Find the largest intersecting families of $k$-subsets of $\{1,\dots,n\}$ for $(n,k)=(5,2),(6,3),(7,3)$ by an exact clique search, and compare with $\binom{n-1}{k-1}$ (Theorem 2.18). For $(6,3)$, is the star the only extremal family?

## Solutions

### P2.1
**How to find it.** Kahn's algorithm keeps the number of remaining predecessors of each element; the elements with count $0$ are the minimal ones, as in the proof of Proposition 2.2. Linear extensions: if $S$ is the set of elements already placed, the next one can be any minimal element of $P\setminus S$, so $e(S)=\sum_{x}e(S\cup\{x\})$ over such $x$, a recursion on $2^n$ subsets.

In [ ]:
def random_poset(n, p):
    less = [[False] * n for _ in range(n)]
    for i in range(n):
        for j in range(i + 1, n):
            less[i][j] = random.random() < p
    for k in range(n):                       # transitive closure (Warshall)
        for i in range(n):
            if less[i][k]:
                for j in range(n):
                    if less[k][j]: less[i][j] = True
    return less

def kahn(less):
    n = len(less)
    preds = [sum(less[i][j] for i in range(n)) for j in range(n)]
    queue = deque(j for j in range(n) if preds[j] == 0); out = []
    while queue:
        x = queue.popleft(); out.append(x)
        for y in range(n):
            if less[x][y]:
                preds[y] -= 1
                if preds[y] == 0: queue.append(y)
    return out

def count_linear_extensions(less):
    n = len(less)
    pred_mask = [sum(1 << i for i in range(n) if less[i][j]) for j in range(n)]
    @lru_cache(None)
    def e(S):
        if S == (1 << n) - 1: return 1
        return sum(e(S | 1 << x) for x in range(n) if not S >> x & 1 and pred_mask[x] & ~S == 0)
    return e(0)

ok = True
for _ in range(100):
    P = random_poset(12, 0.2); order = kahn(P); pos = {x: k for k, x in enumerate(order)}
    ok &= all(pos[i] < pos[j] for i in range(12) for j in range(12) if P[i][j])
print('Kahn gives a linear extension on 100 random posets:', ok)

for n in range(1, 8):
    cells = [(r, c) for r in (1, 2) for c in range(1, n + 1)]
    less = [[a != b and a[0] <= b[0] and a[1] <= b[1] for b in cells] for a in cells]
    print(n, count_linear_extensions(less), math.comb(2 * n, n) // (n + 1))

### P2.2
**How to find it.** A chain cover with $k$ chains uses $|P|-k$ "successor" links $x\to y$ (each chain of length $\ell$ has $\ell-1$ of them), and the links form a matching in the bipartite graph: each element has at most one successor and at most one predecessor. Conversely a matching defines chains by following the links. So minimum chain cover $=|P|-$ maximum matching. Kuhn's algorithm grows the matching along augmenting paths.

In [ ]:
def max_matching(nl, nr, adj):
    # Kuhn's augmenting path algorithm; adj[u] = right neighbours of left vertex u.
    match_r = [-1] * nr
    def augment(u, seen):
        for v in adj[u]:
            if not seen[v]:
                seen[v] = True
                if match_r[v] == -1 or augment(match_r[v], seen):
                    match_r[v] = u; return True
        return False
    size = sum(augment(u, [False] * nr) for u in range(nl))
    return size, match_r

def chain_cover(less):
    n = len(less)
    adj = [[y for y in range(n) if less[x][y]] for x in range(n)]
    size, match_r = max_matching(n, n, adj)
    succ = {match_r[y]: y for y in range(n) if match_r[y] != -1}
    starts = [x for x in range(n) if x not in set(succ.values())]
    chains = []
    for s in starts:
        c = [s]
        while c[-1] in succ: c.append(succ[c[-1]])
        chains.append(c)
    return chains

def width_brute(less):
    n = len(less); best = []
    for mask in range(1 << n):
        S = [i for i in range(n) if mask >> i & 1]
        if len(S) > len(best) and all(not less[a][b] and not less[b][a] for a, b in itertools.combinations(S, 2)):
            best = S
    return best

ok = 0
for _ in range(200):
    P = random_poset(11, random.choice([0.1, 0.2, 0.35]))
    chains = chain_cover(P)
    assert sorted(x for c in chains for x in c) == list(range(11))
    assert all(P[c[i]][c[i + 1]] for c in chains for i in range(len(c) - 1))
    ok += len(chains) == len(width_brute(P))
print('Dilworth confirmed on', ok, 'of 200 random posets')
P = random_poset(11, 0.2)
print('example: chains', chain_cover(P), ' antichain', width_brute(P))

### P2.3
**How to find it.** The tops of the piles increase from left to right (a new card goes on the leftmost pile with a larger top, so it is larger than the tops to its left). Each card on pile $k$ can point to the top of pile $k-1$ at the time it was placed, which is smaller and earlier: following the pointers from any card of the last pile gives an increasing subsequence with one card per pile. Conversely, an increasing subsequence uses each pile at most once, since cards on the same pile decrease. So the number of piles is the LIS. The pile index of a card is exactly the label $r$ of Mirsky's proof.

In [ ]:
def lis(seq):
    tops, idx, back = [], [], [None] * len(seq)
    for i, x in enumerate(seq):
        k = bisect.bisect_left(tops, x)
        if k == len(tops): tops.append(x); idx.append(i)
        else: tops[k] = x; idx[k] = i
        back[i] = idx[k - 1] if k > 0 else None
    out, i = [], idx[-1]
    while i is not None: out.append(seq[i]); i = back[i]
    return out[::-1]

def lds(seq): return [-x for x in lis([-x for x in seq])]

for n in (3, 5, 10, 30):
    worst = min(max(len(lis(s)), len(lds(s))) for s in (random.sample(range(10**6), n * n + 1) for _ in range(300)))
    print(f'n={n}: every one of 300 sequences of length {n*n+1} has a monotone subsequence of length >= {worst} (theorem: {n+1})')

sizes = [10**k for k in range(2, 7)] + [3 * 10**k for k in range(2, 6)]
sizes.sort(); ratios = []
for n in sizes:
    r = np.mean([len(lis(list(np.random.permutation(n)))) for _ in range(5 if n < 10**5 else 1)])
    ratios.append(r / math.sqrt(n))
plt.figure(figsize=(6, 3)); plt.semilogx(sizes, ratios, 'o-'); plt.axhline(2, color='gray', ls=':')
plt.xlabel('n'); plt.ylabel('LIS / sqrt(n)'); plt.title('Longest increasing subsequence of a random permutation'); plt.show()
print([round(x, 3) for x in ratios])

The ratio creeps up towards $2$: the longest increasing subsequence of a random permutation is $2\sqrt n+O(n^{1/6})$, proved by Vershik and Kerov and by Logan and Shepp (1977). The correction is negative and of order $n^{1/6}$, which is why the convergence is so slow; Baik, Deift and Johansson (1999) showed that the fluctuations follow the Tracy–Widom law of random matrix theory (Tome 11). Erdős–Szekeres says $\max(\mathrm{LIS},\mathrm{LDS})\ge\sqrt n$ always; a random permutation achieves about twice that, in both directions at once.

### P2.4
**How to find it.** Represent relations as sets of pairs. Naive iteration recomputes $\mathrm{parent}\circ X$ from scratch at each round. Semi-naive iteration notices that a pair new at round $k+1$ must use a pair new at round $k$, so only the new pairs need to be joined.

In [ ]:
N = 2000
parent = {(random.randrange(i), i) for i in range(1, N) for _ in range(2)}    # each person gets up to two parents among earlier ones
children = defaultdict(set)
for a, b in parent: children[a].add(b)

def kleene(semi):
    by_first = defaultdict(set)
    for a, b in parent: by_first[b].add(a)       # parents of b
    X, delta, rounds, work = set(), set(parent), 0, 0
    while delta:
        rounds += 1
        X |= delta
        source = delta if semi else X
        new = set()
        for (b, c) in source:                      # b ancestor of c, a parent of b  =>  a ancestor of c
            for a in by_first[b]:
                work += 1
                if (a, c) not in X: new.add((a, c))
        delta = new
    return X, rounds, work

for semi in (False, True):
    t = time.time(); X, rounds, work = kleene(semi)
    print(f'{"semi-naive" if semi else "naive":10s}: {len(X)} ancestor pairs, {rounds} rounds, {work} joins, {time.time()-t:.2f}s')

Both compute the same least fixed point, in as many rounds as the longest chain of ancestry (Kleene iteration climbs one generation per round), but semi-naive evaluation does far fewer joins: each derivation is done once instead of once per round. Every Datalog engine and every `WITH RECURSIVE` query in SQL uses this trick.

### P2.5
**How to find it.** (a) Simulate the call stack: keep a stack of pending first arguments; $A(m+1,n+1)=A(m,A(m+1,n))$ pushes $m$ and continues with $(m+1,n)$. (b) Pick a random occurrence of `ba` at each step. (c) Represent a tree as nested lists; the rightmost head is found by always going to the last child.

In [ ]:
def ackermann(m, n):
    stack, calls = [m], 0
    while stack:
        m = stack.pop(); calls += 1
        if m == 0: n += 1
        elif n == 0: stack.append(m - 1); n = 1
        else: stack.append(m - 1); stack.append(m); n -= 1
    return n, calls

for m in (2, 3):
    print(f'A({m},n):', [ackermann(m, n) for n in range(7)])

def rewrite_steps(k):
    w, steps = 'b' * k + 'a', 0
    while 'ba' in w:
        positions = [i for i in range(len(w) - 1) if w[i:i + 2] == 'ba']
        i = random.choice(positions); w = w[:i] + 'aab' + w[i + 2:]; steps += 1
    return w == 'a' * 2**k + 'b' * k, steps
print('rewriting from b^k a:', [(k, rewrite_steps(k)) for k in range(1, 9)])

import copy
def hydra_fight(h, limit=10**6):
    step = 0
    while h:
        step += 1
        if step > limit: return None
        path = [h]                                  # walk to the rightmost head
        while path[-1][-1]: path.append(path[-1][-1])
        node = path[-1]; node.pop()                 # cut the head (an empty list)
        if len(path) >= 2:                          # node is not the root: regrow under its parent
            grand = path[-2]
            for _ in range(step): grand.append(copy.deepcopy(node))
    return step

for name, h in [('path of 2 edges', [[[]]]), ('path of 3 edges', [[[[]]]]), ('two branches', [[[]], [[]]]),
                ('fork at depth 2', [[[], []]]), ('path of 4 edges', [[[[[]]]]])]:
    print(f'{name:16s}: {hydra_fight(copy.deepcopy(h))} steps')

The calls of $A(3,n)$ grow like $4^n$. The Hydra that is a path of three edges dies after $37$ cuts, but with four edges in a row the simulation hits our limit of a million steps, although the theorem guarantees that the fight ends. (For that hydra, the exact count is a number far larger than any we could write in the notebook.) This is the trade-off of Chapter 2's termination proofs: they say *that* a process stops, not *when*.

### P2.6
**How to find it.** The recursion of Theorem 2.17 is three lines. For Dedekind numbers, a monotone $f$ on $B_n$ is determined by $f_0(A)=f(A)$ and $f_1(A)=f(A\cup\{n\})$ for $A\subseteq\{1,\dots,n-1\}$; both are monotone and $f_0\le f_1$, and every such pair arises. As bit masks of length $2^{n-1}$, $f_0\le f_1$ means `f0 & ~f1 == 0`, which numpy checks for all $f_1$ at once.

In [ ]:
def scd(n):
    chains = [[0, 1]]                       # subsets as bit masks; bit k-1 for element k
    for k in range(2, n + 1):
        b = 1 << (k - 1); new = []
        for c in chains:
            new.append(c + [c[-1] | b])
            if len(c) > 1: new.append([a | b for a in c[:-1]])
        chains = new
    return chains

pc = lambda x: bin(x).count('1')
for n in range(1, 13):
    ch = scd(n)
    elems = sorted(a for c in ch for a in c)
    ok = elems == list(range(2**n))
    ok &= all(pc(c[i + 1]) == pc(c[i]) + 1 and c[i] & c[i + 1] == c[i] for c in ch for i in range(len(c) - 1))
    ok &= all(pc(c[0]) + pc(c[-1]) == n for c in ch)
    print(n, len(ch), math.comb(n, n // 2), ok)

def monotone_functions(n):
    if n == 0: return np.array([0, 1], dtype=np.uint64)   # constant 0 and 1 on the single point
    prev = monotone_functions(n - 1); half = 2**(n - 1)
    out = []
    for f0 in prev:
        f1s = prev[(f0 & ~prev) == 0]
        out.append(f0 | (f1s << np.uint64(half)))
    return np.concatenate(out)

for n in range(0, 7):
    t = time.time(); D = len(monotone_functions(n)) if n < 6 else None
    if n == 6:
        prev = monotone_functions(5)
        D = int(sum(((f0 & ~prev) == 0).sum() for f0 in prev))
    print(f'D({n}) = {D}   ({time.time()-t:.2f}s)')

$2, 3, 6, 20, 168, 7581, 7\,828\,354$: the Dedekind numbers. The next ones are $2\,414\,682\,040\,998$ (Church, 1965, for $n=7$), then $n=8$ (Wiedemann, 1991) and $n=9$, about $2.86\times10^{41}$, computed in 2023 independently by Jäkel and by Van Hirtum et al. with supercomputers and clever symmetry arguments. No formula is known.

### P2.7
**How to find it.** (a) Add random sets to an antichain while no inclusion is created. (b) Intersecting families are cliques in the graph whose vertices are the $k$-sets, joined when they meet; a simple branch-and-bound clique search (Bron–Kerbosch with pivoting) is fast for $\binom73=35$ vertices.

In [ ]:
def random_maximal_antichain(n):
    sets = list(range(2**n)); random.shuffle(sets); A = []
    for s in sets:
        if all(s & a != s and s & a != a for a in A): A.append(s)
    return A

for _ in range(5):
    A = random_maximal_antichain(8)
    print(f'antichain of size {len(A):3d}: LYM sum = {sum(1 / math.comb(8, pc(a)) for a in A):.3f}  sizes {sorted(set(pc(a) for a in A))}')
print('middle rank: size', math.comb(8, 4), 'LYM sum 1')

def max_cliques(vertices, adj):
    best = []
    def bk(R, P, X):
        nonlocal best
        if not P and not X:
            if len(R) > len(best[0]) if best else True: best = [R]
            elif len(R) == len(best[0]): best.append(R)
            return
        u = max(P | X, key=lambda v: len(adj[v] & P))
        for v in list(P - adj[u]):
            bk(R + [v], P & adj[v], X & adj[v]); P.remove(v); X.add(v)
    bk([], set(vertices), set())
    return best

for n, k in [(5, 2), (6, 3), (7, 3)]:
    ks = [frozenset(c) for c in itertools.combinations(range(1, n + 1), k)]
    adj = {a: {b for b in ks if b != a and a & b} for a in ks}
    cl = max_cliques(ks, adj)
    stars = sum(1 for c in cl if frozenset.intersection(*c))
    print(f'n={n}, k={k}: maximum {len(cl[0])} (bound {math.comb(n - 1, k - 1)}), {len(cl)} maximum families, {stars} of them stars')

Random maximal antichains mix several ranks and stay well below the bound; the LYM sum reaches $1$ only for full ranks. For $n>2k$ the stars are the only extremal intersecting families (Hilton and Milner, 1967), but for $n=2k$ there are many more: any family containing exactly one of each complementary pair $\{A,\bar A\}$ is intersecting and has $\frac12\binom{2k}k=\binom{2k-1}{k-1}$ members. The search above counts them for $(6,3)$.